# 01 · Fase 1 — MVP: chain executável com entrada e saída estruturadas

**Projeto ApuraCota** — agente de apoio à apuração jornalística dos gastos da Cota para o Exercício da Atividade Parlamentar (CEAP) da Câmara dos Deputados.

Este notebook cobre as rubricas **1.1 a 1.6**: anatomia do agente, arquitetura de planejamento e memória, templates de prompt, output parser JSON, interface Runnable (`invoke`/`batch`/`stream`), composição imperativa com `@chain` e declarativa com LCEL.

> **Sobre o modo offline.** O enunciado pede alternativas *dummy* para APIs e credenciais. Sem `OPENAI_API_KEY` (ou com `MODO_OFFLINE=true`), o projeto troca o GPT por um **modelo roteirizado** (`src/llm.py`: regras determinísticas que escolhem a ferramenta e formatam a observação), os embeddings por `DeterministicFakeEmbedding` e as APIs por **fixtures gravados** de chamadas reais (`fixtures/`). A estrutura das chains, do agente, da memória e da verificação é exatamente a mesma; só a "inteligência" muda. As saídas deste notebook foram geradas no modo indicado na célula de configuração.

## 1.1 Anatomia e arquitetura cognitiva do agente

**Problema.** Repórteres precisam responder rápido a perguntas como "quanto o deputado X gastou com combustível em 2025?", "isso está acima da média do partido?" ou "a cota pode pagar hospedagem em Brasília?". Hoje isso exige baixar CSVs, consultar a API da Câmara e ler o Ato da Mesa 43/2009. O ApuraCota é um agente que faz esse trabalho sob demanda, com fonte e período em cada resposta.

**Anatomia (os cinco componentes da aula 1):**

| Componente | No ApuraCota |
|---|---|
| **LLM** (motor cognitivo) | `gpt-4o-mini` no agente (custo), `gpt-4o` como juiz da verificação/avaliação. Offline: modelo roteirizado. |
| **System prompt** | PTCF (Persona, Task, Context, Format) em `src/prompts.py`: analista de dados da redação, regras de roteamento, "nunca invente valores". |
| **Memória** | Curto prazo: turnos da apuração (checkpointer por `thread_id`). Longo prazo: regras da cota (semântica), apurações anteriores (episódica), prompt e docstrings (procedural). |
| **Ferramentas** | `buscar_deputado`, `listar_despesas` e `dados_fornecedor` (APIs externas), `consultar_ceap` (text-to-SQL), `calcular_estatisticas` (hardcoded), `consultar_regras_cota` (RAG), `gravar_apuracao`/`lembrar_apuracao` (memória). |
| **Base de conhecimento** | Ato da Mesa 43/2009 consolidado em índice FAISS; CSV anual da CEAP em SQLite. |

**Classificação.** Arquitetura **híbrida**: reativa para perguntas diretas (uma ferramenta e resposta) e deliberativa para comparações, que exigem planejar várias chamadas e verificar os números antes de responder. **Maturidade agêntica**: no início da fase 1 é um *agente de busca* (tipo 1); ao fim da fase 5 vira um *agente de tarefa com camada de verificação* (tipo 2 com V&V), ainda sem autonomia para agir no mundo (não publica nada, só apura).

## 1.2 Arquitetura de planejamento e memória aumentada

**Decomposição de uma tarefa complexa.** "Compare o gasto da Tabata Amaral com combustível com a média do partido dela" vira o plano:

1. `buscar_deputado("Tabata Amaral")` → id 204534, PSB-SP (se houver homônimos, perguntar antes de seguir);
2. `listar_despesas(204534, 2025, categoria="combustível")` → total do deputado;
3. `consultar_ceap(SELECT AVG(...) WHERE partido='PSB' ...)` → média por deputado do partido;
4. `calcular_estatisticas([media, gasto], "variacao_pct")` → diferença percentual (a LLM nunca faz conta de cabeça);
5. verificar os números contra a fonte autoritativa (fase 5) e responder com período e fonte.

**Memória aumentada (taxonomia CoALA, detalhada na fase 3).** O plano acima só funciona em conversa porque o turno 2 ("e com passagens aéreas?") herda deputado e ano do turno 1 (memória de curto prazo), e "aquele fornecedor que vimos semana passada" depende de apurações gravadas com data (memória episódica de longo prazo).

In [1]:
# Configuração comum: roda a partir da raiz do repositório e mostra o modo de execução
import json
import os
import sys
from importlib.metadata import version
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(RAIZ)
sys.path.insert(0, str(RAIZ))
import warnings

warnings.filterwarnings("ignore")

from src import config, llm

print(
    "LangChain",
    version("langchain"),
    "| langchain-core",
    version("langchain-core"),
    "| LangGraph",
    version("langgraph"),
    "| LangSmith",
    version("langsmith"),
)
print(
    "Modo:",
    "OFFLINE (modelo dummy roteirizado + fixtures)" if llm.offline() else f"ONLINE ({config.MODELO_AGENTE})",
)

LangChain 1.4.9 | langchain-core 1.7.6 | LangGraph 1.2.20 | LangSmith 0.14.17
Modo: OFFLINE (modelo dummy roteirizado + fixtures)


## 1.3 Templates reutilizáveis: `PromptTemplate` e `ChatPromptTemplate`

`PromptTemplate` serve para uma chamada de texto simples; `ChatPromptTemplate` organiza mensagens por papel (system/human), que é o formato que os modelos de chat esperam.

In [2]:
from src import prompts

print("--- PromptTemplate (texto simples) ---")
print(prompts.PROMPT_NORMALIZAR_NOME.format(texto="Tabata Amaral (PSB-SP)"))

print("\n--- ChatPromptTemplate (mensagens por papel) ---")
for m in prompts.PROMPT_PEDIDO_APURACAO.format_messages(
    pergunta="Quanto a Tabata Amaral gastou com combustível em 2025?"
):
    print(f"[{m.type}] {m.content}")

--- PromptTemplate (texto simples) ---
Extraia nome, partido e UF do texto a seguir. Se faltar algum, use null.
Texto: Tabata Amaral (PSB-SP)
Responda apenas em JSON com as chaves nome, partido, uf.

--- ChatPromptTemplate (mensagens por papel) ---
[system] Voce e um analista de dados de uma redacao especializada na cota parlamentar (CEAP). Transforme a pergunta do reporter em um pedido estruturado. Se faltar deputado ou periodo para uma consulta de gasto, preencha dado_faltante.
[human] Quanto a Tabata Amaral gastou com combustível em 2025?


## 1.4 Output parser para saída JSON

A classe Pydantic `PedidoApuracao` é o **contrato** entre a pergunta livre e o resto do sistema. Dois caminhos: `with_structured_output` (function calling nativo, preferido) e `PydanticOutputParser` (JSON explícito no prompt, útil para mostrar o schema e para modelos sem function calling).

In [3]:
from src.chains import PedidoApuracao, chain_pedido, parser_pedido

print(json.dumps(PedidoApuracao.model_json_schema()["properties"], ensure_ascii=False, indent=1)[:900], "...")
print("\n--- format_instructions do PydanticOutputParser (trecho) ---")
print(parser_pedido.get_format_instructions()[:400], "...")

{
 "intencao": {
  "enum": [
   "consultar_gasto",
   "comparar",
   "ranking_fornecedor",
   "regra_cota",
   "resumo",
   "conversa"
  ],
  "title": "Intencao",
  "type": "string"
 },
 "deputado": {
  "anyOf": [
   {
    "type": "string"
   },
   {
    "type": "null"
   }
  ],
  "default": null,
  "description": "Nome do deputado citado.",
  "title": "Deputado"
 },
 "partido": {
  "anyOf": [
   {
    "type": "string"
   },
   {
    "type": "null"
   }
  ],
  "default": null,
  "title": "Partido"
 },
 "uf": {
  "anyOf": [
   {
    "type": "string"
   },
   {
    "type": "null"
   }
  ],
  "default": null,
  "title": "Uf"
 },
 "ano": {
  "anyOf": [
   {
    "type": "integer"
   },
   {
    "type": "null"
   }
  ],
  "default": null,
  "title": "Ano"
 },
 "mes": {
  "anyOf": [
   {
    "type": "integer"
   },
   {
    "type": "null"
   }
  ],
  "default": null,
  "title": "Mes"
 },
 "cate ...

--- format_instructions do PydanticOutputParser (trecho) ---
The output should be formatted as

In [4]:
pedido = chain_pedido().invoke({"pergunta": "Quanto a Tabata Amaral gastou com combustível em 2025?"})
print(type(pedido).__name__, "->", pedido.model_dump())

PedidoApuracao -> {'intencao': 'consultar_gasto', 'deputado': 'Tabata Amaral', 'partido': None, 'uf': None, 'ano': 2025, 'mes': None, 'categoria': 'COMBUST', 'dado_faltante': None}


## 1.5 Interface Runnable: `invoke`, `batch`, `stream` e composição imperativa com `@chain`

In [5]:
perguntas = [
    "Quanto a Tabata Amaral gastou com combustível em 2025?",
    "Compare o gasto da Tabata Amaral com a média do PSB.",
    "Gastos do Silva em 2025",
    "A cota pode pagar hotel em Brasília?",
    "Quais os maiores fornecedores do Nikolas Ferreira em 2025?",
]
chain = chain_pedido()

print("--- invoke ---")
print(chain.invoke({"pergunta": perguntas[0]}))
print("\n--- batch (5 perguntas em paralelo) ---")
for p, r in zip(perguntas, chain.batch([{"pergunta": p} for p in perguntas])):
    print(
        f"{r.intencao:18} deputado={r.deputado!s:16} ano={r.ano} categoria={r.categoria} faltante={r.dado_faltante}"
    )

--- invoke ---
intencao='consultar_gasto' deputado='Tabata Amaral' partido=None uf=None ano=2025 mes=None categoria='COMBUST' dado_faltante=None

--- batch (5 perguntas em paralelo) ---
consultar_gasto    deputado=Tabata Amaral    ano=2025 categoria=COMBUST faltante=None
comparar           deputado=Tabata Amaral    ano=None categoria=None faltante=None
consultar_gasto    deputado=Silva            ano=2025 categoria=None faltante=None
regra_cota         deputado=Brasília         ano=None categoria=None faltante=None
ranking_fornecedor deputado=Nikolas Ferreira ano=2025 categoria=None faltante=None


In [6]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

prompt_resumo = ChatPromptTemplate.from_messages(
    [
        ("system", "Resuma em uma frase, para um repórter, o pedido a seguir."),
        ("human", "{pergunta}"),
    ]
)
chain_texto = prompt_resumo | llm.chat_model() | StrOutputParser()
print("--- stream (tokens chegando) ---")
for pedaco in chain_texto.stream({"pergunta": perguntas[1]}):
    print(pedaco, end="", flush=True)
print()

--- stream (tokens chegando) ---
Resumo (offline): Compare o gasto da Tabata Amaral com a média do PSB.

In [7]:
from src.chains import pedido_ou_esclarecimento  # função decorada com @chain

print("Caso completo  ->", pedido_ou_esclarecimento.invoke(perguntas[0]))
print("Falta deputado ->", pedido_ou_esclarecimento.invoke("Quanto gastou com combustível em 2025?"))

Caso completo  -> intencao='consultar_gasto' deputado='Tabata Amaral' partido=None uf=None ano=2025 mes=None categoria='COMBUST' dado_faltante=None
Falta deputado -> Para seguir preciso saber: deputado. Pode informar?


## 1.6 Declarativo (LCEL) ou imperativo (`@chain`)? Decisão e justificativa

- **LCEL** (`prompt | llm | parser`) é o padrão do projeto. Motivo técnico, não de estilo: cada composição LCEL é um `Runnable` e ganha **de graça** `batch` (paralelismo), `stream`, execução assíncrona e **rastreamento no LangSmith** (cada etapa vira um span). Isso é o que permite, na fase 5, avaliar a trajetória sem instrumentar nada à mão.
- **Imperativo com `@chain`** fica reservado aos pontos com **lógica condicional**: em `pedido_ou_esclarecimento`, se `dado_faltante` estiver preenchido, devolvemos uma pergunta em vez de seguir o pipeline. Em LCEL isso exigiria `RunnableBranch`, menos legível para quem for manter o código. A função decorada continua sendo um `Runnable`, então também aparece no tracing.

**Resultado da fase 1:** uma chain que transforma a pergunta livre em um `PedidoApuracao` validado, já testada em lote com cinco perguntas de perfis diferentes (gasto direto, comparação, homônimo, regra e ranking).